In [ ]:
#25년 데이터 적발건수가 1,11월에 집중되어있음 월별데이터는 참고만하는게 좋을것같음

import pandas as pd
import os

# 분석에 사용할 원본 CSV 파일 경로를 지정
file_2025 = "../data/대구광역시 북구_불법주정차위반정보_20260102.csv"

# CSV 파일을 UTF-8(BOM 포함) 인코딩으로 불러옵니다.
df = pd.read_csv(file_2025, encoding="utf-8-sig")

# 데이터가 정상적으로 불러와졌는지 확인합니다.
print(df.head())

# 컬럼명과 데이터 크기를 확인합니다.
print("20260102 컬럼:", df.columns.tolist())
print("데이터 크기:", df.shape)

# 2017~2023 데이터와 비교하면 위반시간, 위반장소 컬럼이 없습니다.
print("위반시간 컬럼 있음:", "위반시간" in df.columns)
print("위반장소 컬럼 있음:", "위반장소" in df.columns)

# 위반일시 범위를 확인합니다.
date_check = pd.to_datetime(df["위반일시"], errors="coerce")
print("20260102 위반일시 범위:", date_check.min(), "~", date_check.max())

# 날짜로 바뀌지 않는 값이 있는지 확인합니다.
print("날짜 변환 실패 행 수:", date_check.isnull().sum())

# '단속구분'에 어떤 값이 있는지 확인합니다.
print(df["단속구분"].value_counts(dropna=False))

# 참고용으로 '위반사항' 값도 확인합니다.
print()
print(df["위반사항"].value_counts(dropna=False))
# 위반일시를 날짜형으로 변환합니다.
df["위반일시"] = pd.to_datetime(df["위반일시"], errors="coerce")

# 과태료부과일자도 날짜형으로 변환합니다.
df["과태료부과일자"] = pd.to_datetime(df["과태료부과일자"], errors="coerce")

# 단속구분 앞뒤 공백을 제거합니다.
df["단속구분"] = df["단속구분"].astype(str).str.strip()

# 날짜 분석에 사용할 연도와 월 컬럼을 만듭니다.
df["연도"] = df["위반일시"].dt.year
df["월"] = df["위반일시"].dt.month

# 숫자로 계산되는 요일을 한글 요일로 바꾸기 위한 딕셔너리입니다.
weekday_map = {
    0: "월요일",
    1: "화요일",
    2: "수요일",
    3: "목요일",
    4: "금요일",
    5: "토요일",
    6: "일요일"
}

# 위반일시에서 요일을 계산하여 새 컬럼을 만듭니다.
df["요일"] = df["위반일시"].dt.dayofweek.map(weekday_map)

# 주요 분석 컬럼에 결측치가 있는지 확인합니다.
print(df[["위반일시", "단속구분", "연도", "월", "요일"]].isnull().sum())

# 같은 값이 반복되는 행의 개수도 확인합니다.
# 이 데이터에는 차량번호, 시간, 장소가 없어서 같은 날 같은 유형의 여러 단속이 똑같이 보입니다.
# 실제로 다른 단속이므로 삭제하지 않습니다.
print("완전히 동일한 행 수:", df.duplicated().sum())



In [ ]:
#월별데이터 확인주의
#특정달에만 단속이몰려있음

# 월별 행 수를 확인합니다.
print(df.groupby("월").size().reindex(range(1, 13), fill_value=0))

# 과태료부과일자 범위도 확인합니다.
print("\n과태료부과일자 범위:", df["과태료부과일자"].min(), "~", df["과태료부과일자"].max())

# 최종 전처리 결과를 확인합니다.
print("최종 데이터 크기:", df.shape)
print("위반일시 범위:", df["위반일시"].min(), "~", df["위반일시"].max())
print("연도별 행 수")
print(df.groupby("연도").size())

# 전처리 결과를 저장할 폴더가 없으면 새로 만듭니다.
os.makedirs("../data/processed", exist_ok=True)

# 02_date_analysis_2025에서 사용할 데이터를 저장합니다.
# utf-8-sig로 저장하면 엑셀에서 열었을 때도 한글이 깨질 가능성이 적습니다.
output_path = "../data/processed/violation_2025.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)



In [ ]:
#시각화 부분 시작

In [ ]:
# 그래프를 그리기 위한 matplotlib와 seaborn을 불러옵니다.
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 파일을 저장하기 위해 os를 불러옵니다.
import os

# 윈도우에서 그래프 한글이 깨지지 않도록 맑은 고딕을 사용합니다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 그래프를 저장할 폴더가 없으면 만듭니다.
os.makedirs("images", exist_ok=True)


In [ ]:
# 월별 불법주정차 건수를 계산합니다. (건수가 없는 월은 0으로 채웁니다)
month_count = df.groupby("월").size().reindex(range(1, 13), fill_value=0)

# 계산 결과를 확인합니다.
print(month_count)

# 1월부터 12월까지의 건수를 막대그래프로 비교합니다.
plt.figure(figsize=(10, 5))
plt.bar(month_count.index, month_count.values)
plt.title("2025년 월별 불법주정차 단속 건수")
plt.xlabel("월")
plt.ylabel("단속 건수")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.savefig("images/date2025_month_count.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 월요일부터 일요일까지 원하는 순서로 정렬하기 위한 리스트입니다.
weekday_order = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]

# 요일별 불법주정차 건수를 계산하고 월~일 순서로 정렬합니다.
weekday_count = df.groupby("요일").size().reindex(weekday_order)

# 계산 결과를 확인합니다.
print(weekday_count)

# 요일별 건수를 막대그래프로 비교합니다.
plt.figure(figsize=(9, 5))
plt.bar(weekday_count.index, weekday_count.values)
plt.title("2025년 요일별 불법주정차 단속 건수")
plt.xlabel("요일")
plt.ylabel("단속 건수")
plt.tight_layout()
plt.savefig("images/date2025_weekday_count.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 단속구분별 건수를 많은 순으로 계산합니다.
type_count = df.groupby("단속구분").size().sort_values(ascending=False)

# 건수와 비율(%)을 함께 확인합니다.
print(type_count)
print()
print((type_count / len(df) * 100).round(1))

# 단속구분별 건수를 막대그래프로 비교합니다.
plt.figure(figsize=(9, 5))
plt.bar(type_count.index, type_count.values)
plt.title("2025년 단속구분별 불법주정차 단속 건수")
plt.xlabel("단속구분")
plt.ylabel("단속 건수")
plt.tight_layout()
plt.savefig("images/date2025_type_count.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 월과 단속구분을 기준으로 단속 건수를 교차 집계합니다.
month_type = df.groupby(["월", "단속구분"]).size().unstack(fill_value=0)
month_type = month_type.reindex(range(1, 13), fill_value=0)[type_count.index]

# 교차표를 확인합니다.
display(month_type)

# 월별 단속구분 구성을 누적 막대그래프로 나타냅니다.
month_type.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("2025년 월 × 단속구분 불법주정차 단속 건수")
plt.xlabel("월")
plt.ylabel("단속 건수")
plt.xticks(rotation=0)
plt.legend(title="단속구분")
plt.tight_layout()
plt.savefig("images/date2025_month_type.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 요일과 단속구분을 기준으로 단속 건수를 교차 집계합니다.
weekday_type = df.groupby(["요일", "단속구분"]).size().unstack(fill_value=0)
weekday_type = weekday_type.reindex(weekday_order)[type_count.index]

# 교차표를 확인합니다.
display(weekday_type)

# 요일별 단속구분 패턴을 한눈에 보기 위해 히트맵으로 나타냅니다.
plt.figure(figsize=(9, 6))
sns.heatmap(weekday_type, cmap="YlOrRd", annot=True, fmt="d")
plt.title("2025년 요일 × 단속구분 불법주정차 단속 건수")
plt.xlabel("단속구분")
plt.ylabel("요일")
plt.tight_layout()
plt.savefig("images/date2025_weekday_type_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 요일마다 단속구분이 차지하는 비율(%)을 계산합니다.
weekday_type_ratio = weekday_type.div(weekday_type.sum(axis=1), axis=0) * 100

# 비율표를 확인합니다.
display(weekday_type_ratio.round(1))

# 요일별 단속구분 비율을 100% 누적 막대그래프로 나타냅니다.
weekday_type_ratio.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.title("2025년 요일별 단속구분 비율")
plt.xlabel("요일")
plt.ylabel("비율 (%)")
plt.xticks(rotation=0)
plt.legend(title="단속구분", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("images/date2025_weekday_type_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 가장 많은 월, 요일, 단속구분을 찾아 간단하게 출력합니다.
top_month = month_count.idxmax()
top_weekday = weekday_count.idxmax()
top_type = type_count.idxmax()

print("2025년 단속 건수:", len(df), "건")
print("단속 건수가 가장 많은 월:", top_month, "월")
print("단속 건수가 가장 많은 요일:", top_weekday)
print("가장 많은 단속구분:", top_type, f"({type_count.max() / len(df) * 100:.1f}%)")

print("\n※ 이 파일은 과태료부과일자 기준으로 뽑힌 데이터라 월별 건수는 참고용입니다.")
print("※ 장소·시간 정보가 없어 날짜와 단속구분 기준으로만 해석합니다.")